In [ ]:
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.surface_code.merge import MergeSurgery
from library.surface_code.patch import SurfaceCodePatch
from utils.circuit_faults import analyse_heuristic

In [ ]:
DISTANCE = 5
qubits = SurfaceCodePatch.make_array(distance=DISTANCE, counts=(2,1))
circuitry = Circuitry(qubits=qubits)
basis = Pauli.Z
last = DISTANCE - 1

source = SurfaceCodePatch(qubits, distance=DISTANCE, anchor=(1,1))
target = SurfaceCodePatch(qubits, distance=DISTANCE, anchor=(DISTANCE+2, 1))
merger = MergeSurgery(qubits, source, target)

circuitry.annotate_polygons(source.get_polygons())
circuitry.annotate_polygons(target.get_polygons())

for rnd in range(DISTANCE):
    prepare = basis if rnd == 0 else None
    for mmt in SurfaceCodePatch.MOMENTS:
        source.append_round(circuitry, moment=mmt, prepare=prepare, prefix=f"SRC:M0:R{rnd}")
        target.append_round(circuitry, moment=mmt, prepare=prepare, prefix=f"TGT:M0:R{rnd}")
        circuitry.append_tick()

merger.annotate_polygons(circuitry)

merger.append_merge(circuitry)

circuitry.annotate_polygons(source.get_polygons())
circuitry.annotate_polygons(target.get_polygons())

for rnd in range(DISTANCE):
    measure = basis if rnd == last else None
    for mmt in SurfaceCodePatch.MOMENTS:
        source.append_round(circuitry, moment=mmt, measure=measure, prefix=f"SRC:M1:R{rnd}")
        target.append_round(circuitry, moment=mmt, measure=measure, prefix=f"TGT:M1:R{rnd}")
        circuitry.append_tick()

source.annotate_detectors(circuitry, prepared=basis, prefix="SRC:M0")
target.annotate_detectors(circuitry, prepared=basis, prefix="TGT:M0")
merger.annotate_detectors(circuitry, preceding=("SRC:M0", "TGT:M0"), following=("SRC:M1", "TGT:M1"))
source.annotate_detectors(circuitry, measured=basis, prefix="SRC:M1")
target.annotate_detectors(circuitry, measured=basis, prefix="TGT:M1")

circuitry.attach_observable(
    0, f"MRG:JCT:R{last}:D{DISTANCE}",
    *(f"SRC:M1:R{last}:D{q*DISTANCE}" for q in range(DISTANCE)),
    *(f"TGT:M1:R{last}:D{q*DISTANCE}" for q in range(DISTANCE))
)

circuitry.detectors_report()

display(Markdown(f"[Open in Crumble]({circuitry.to_crumble_url(spacing=True)})"))

In [ ]:
analyse_heuristic(circuitry.as_stim)